# Lab 14: Path-integral MCMC

Remember how we used random spin flips to measure properties of the Ising model? Today we'll use the same sampling idea for a quantum oscillator.

We'll build a position histogram from sampled paths and compare it with an answer we already know. Then we'll work out how to improve the calculation.

PHYS 142/242 · Friday, November 13, 2026

## Remember the Ising model?

We saved whole spin lattices and measured their magnetization. Today we'll save whole paths and measure their positions.

| What we did for Ising | What we'll do here |
|---|---|
| Store every spin in a lattice | Store every position along a path |
| Propose a spin flip | Propose moving one position |
| Use the energy change to accept or reject | Use the action change to accept or reject |
| Measure magnetization from saved lattices | Build a position histogram from saved paths |

Let's start with the answer we want, then look at what a path contains.

## Set up the notebook

Run the cells in order. Each simulation starts a fresh chain from a stated seed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
from IPython.display import HTML, display

plt.rcParams.update({
    "font.size": 12, "figure.figsize": (6.2, 3.6),
    "figure.dpi": 110, "figure.constrained_layout.use": True,
})
M, OMEGA, HBAR = 1.0, 1.0, 1.0
TAU = 30.0

NumPy holds the paths; Matplotlib draws them. Numba compiles the repeated updates. We use units with $m=\omega=\hbar=1$.

## What are we trying to calculate?

For a harmonic oscillator, $V(x)=m\omega^2x^2/2$. We want the probability density for a position measurement in its ground state:

$$\rho_0(x)=|\psi_0(x)|^2=\sqrt{\frac{m\omega}{\pi\hbar}}\,e^{-m\omega x^2/\hbar},
\qquad \langle x^2\rangle_0=\frac{\hbar}{2m\omega}=\frac12.$$

We know the answer here, so we can check whether our sampling works.

In [ ]:
x_grid = np.linspace(-3, 3, 301)
rho_ground = np.sqrt(M * OMEGA / (np.pi * HBAR)) * np.exp(
    -M * OMEGA * x_grid**2 / HBAR)
fig, ax = plt.subplots()
ax.plot(x_grid, rho_ground, color="#007c91", lw=2)
ax.set(xlabel="Position x", ylabel="Probability density")
plt.show()

Our target is this curve and its mean-square position, $\langle x^2\rangle=0.5$. Collecting positions from many sampled paths will give us the histogram to compare with it.

## A simpler example: paths for a free particle

For this aside, set $V(x)=0$: a **free particle**. Let's see what we mean by a path before adding the oscillator potential.

Remember that one Ising configuration contained every spin. Here **one configuration contains every position along a curve**, $x=(x_0,\ldots,x_{N_\tau-1})$.

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_free_rng = np.random.default_rng(1401)
lab14_free_times = np.linspace(0, 4, 41)
lab14_free_dt = lab14_free_times[1]
lab14_free_steps = lab14_free_rng.normal(
    0, np.sqrt(HBAR * lab14_free_dt / M), size=(8, 40))
lab14_free_paths = np.c_[np.zeros(8), np.cumsum(lab14_free_steps, axis=1)]
fig, ax = plt.subplots(figsize=(11.2, 3.0))
for path in lab14_free_paths[1:]:
    ax.plot(lab14_free_times, path, color='#b4c2d0', lw=1.2)
ax.plot(lab14_free_times, lab14_free_paths[0], 'o-', ms=3,
        color='#007c91', lw=2, label='One configuration = this entire path')
point = 23
ax.scatter(lab14_free_times[point], lab14_free_paths[0, point],
           color='#a54632', s=65, zorder=5)
ax.annotate(r'$x_i$: one coordinate',
    (lab14_free_times[point], lab14_free_paths[0, point]),
    xytext=(0.56, 0.12), textcoords='axes fraction',
    arrowprops={'arrowstyle': '->', 'color': '#a54632'}, color='#a54632')
ax.set(xlabel=r'Imaginary time $\tau$', ylabel='Position x', xlim=(0, 4))
ax.legend(loc='upper left', framealpha=0.95, fontsize=11)
plt.show()

Each dot gives the position at one imaginary-time point. The blue curve is one whole configuration; the gray curves are other possibilities. These examples start at zero and have free endpoints. Imaginary time labels the curve; Monte Carlo steps label changes from one curve to another.

## Stay with the free particle: many possible paths

Now fix both endpoints at zero. There are still many different curves that connect them. These panels show one, a few, and many sampled free-particle paths.

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_bridge_rng = np.random.default_rng(1402)
lab14_bridge_steps = lab14_bridge_rng.normal(
    0, np.sqrt(HBAR * lab14_free_dt / M), size=(100, 40))
lab14_bridge_walks = np.c_[np.zeros(100), np.cumsum(lab14_bridge_steps, axis=1)]
lab14_bridges = lab14_bridge_walks - (
    lab14_free_times / lab14_free_times[-1]) * lab14_bridge_walks[:, -1, None]
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.0), sharex=True, sharey=True)
for ax, count in zip(axes, [1, 5, 100]):
    for path in lab14_bridges[:count]:
        ax.plot(lab14_free_times, path, color='#7daab8',
                alpha=0.22 if count == 100 else 0.7, lw=1)
    ax.plot(lab14_free_times, lab14_bridges[0], color='#007c91', lw=2)
    ax.scatter([0, 4], [0, 0], color='#a54632', s=30, zorder=5)
    ax.set(title=f'{count} path' + ('s' if count > 1 else ''),
           xlabel=r'Imaginary time $\tau$', xlim=(0, 4))
axes[0].set_ylabel('Position x')
plt.show()

Every curve is a whole configuration. Fixing the endpoints lets us see how much the middle can vary. Next we'll restore the oscillator potential and let the common endpoint vary too.

## Back to the oscillator: close the path

Restore $V(x)=m\omega^2x^2/2$. To calculate equilibrium properties, we use closed paths: $x_{N_\tau}=x_0$.

We sample the common endpoint along with all the other positions. The first and last stored points are neighbors.

In [ ]:
dt = 1.0
n_tau = int(round(TAU / dt))
tau_grid = np.arange(n_tau + 1) * dt
example_path = 0.8 * np.sin(2 * np.pi * np.arange(n_tau) / n_tau)
fig, ax = plt.subplots()
ax.plot(tau_grid, np.r_[example_path, example_path[0]], "o-", ms=3)
ax.set(xlabel=r"Imaginary time $\tau$", ylabel="Position x")
plt.show()

This example shows how closure works. We repeat the first coordinate at the end of the drawing; the array stores each coordinate once. The closing link joins the last stored point to the first.

## Why make the imaginary-time interval long?

Remember how cooling the Ising model changed which configurations we sampled? Here the temperature is set by $\tau_{\rm total}=\hbar/(k_B T)$.

**Longer imaginary time means lower temperature.** The excited-state populations shrink and the ground state dominates.

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_b = np.linspace(0.1, 30, 600)
lab14_q = np.exp(-lab14_b)
fig, axes = plt.subplots(1, 2, figsize=(11.2, 3.0))
axes[0].plot(lab14_b, 1-lab14_q, color='#007c91', lw=2, label='Ground state')
axes[0].plot(lab14_b, (1-lab14_q)*lab14_q, color='#a54632', lw=2, label='First excited')
axes[0].plot(lab14_b, lab14_q**2, color='#8a7098', lw=2, label='All higher states')
axes[0].axvline(30, color='#172b4d', ls=':', lw=1.5)
axes[0].set(xlabel=r'Total imaginary time $\omega\tau_{\rm total}$',
            ylabel='Fraction in each state', xlim=(0, 31), ylim=(0, 1.06))
axes[0].legend(loc='center right', fontsize=10)
thermal_x = np.linspace(-3.8, 3.8, 401)
for interval, color in [(1, '#8a7098'), (3, '#a54632'), (30, '#007c91')]:
    variance = HBAR / (2*M*OMEGA*np.tanh(interval/2))
    density = np.exp(-thermal_x**2/(2*variance))/np.sqrt(2*np.pi*variance)
    axes[1].plot(thermal_x, density, color=color, lw=2,
                 label=rf'$\omega\tau_{{\rm total}}={interval}$')
axes[1].plot(thermal_x, np.sqrt(M*OMEGA/(np.pi*HBAR))*
             np.exp(-M*OMEGA*thermal_x**2/HBAR), '--', color='#172b4d',
             lw=1.5, label='Ground state')
axes[1].set(xlabel='Position x', ylabel='Position density', xlim=(-3.8, 3.8))
axes[1].legend(fontsize=10)
plt.show()

These are continuum oscillator predictions. At $\omega\tau_{\rm total}=30$, the ground state dominates. We also need a small enough time step to represent the path accurately; we'll check that separately.

## What does one link contribute?

Remember adding the neighbor-bond energies in the Ising model? Here we add contributions from neighboring positions along the path:

$$s(x_i,x_{i+1})=\underbrace{\frac{m(x_{i+1}-x_i)^2}{2\Delta\tau}}_{\text{kinetic: penalizes jumps}}+
\underbrace{\Delta\tau\,\frac{m\omega^2x_i^2}{2}}_{\text{potential: penalizes large positions}}.$$

In [ ]:
@njit
def action(x_left, x_right, dt):
    kinetic = 0.5 * M * (x_right - x_left)**2 / dt
    potential = 0.5 * M * OMEGA**2 * x_left**2 * dt
    return kinetic + potential

This function implements those two terms. We evaluate the potential at the left endpoint of each link.

## Add the links to get the path's weight

For Ising, we sampled with weight $e^{-E/(k_BT)}$. For these paths, the action $S_E$ sets the weight:

$$S_E(x)=\sum_{i=0}^{N_\tau-1}s(x_i,x_{i+1}),\qquad x_{N_\tau}=x_0,
\qquad P(x)\propto e^{-S_E(x)/\hbar}.$$

In [ ]:
@njit
def total_action(path, dt):
    result = 0.0
    for i in range(path.size):
        result += action(path[i], path[(i + 1) % path.size], dt)
    return result

The modulo operation includes the closing link. Each link contributes once. A larger action gives a smaller weight, so those paths appear less often in our samples.

## Watch the action change

First pull one point away from its neighbors. Then raise every position by the same amount. Watch what happens to the jump cost, the potential cost, and the relative weight.

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
import numpy as lab14_np
import re as lab14_re
import matplotlib.pyplot as lab14_plt
from matplotlib.animation import FuncAnimation as lab14_FuncAnimation
from IPython.display import HTML as lab14_HTML, display as lab14_display

# This illustration uses its own path and does not advance the MCMC chain.
lab14_action_m = lab14_action_omega = lab14_action_hbar = 1.0
lab14_action_dt = 1.0
lab14_action_n = 30
lab14_action_index = lab14_action_n // 2
lab14_action_tau = lab14_np.arange(lab14_action_n + 1) * lab14_action_dt
lab14_action_reference_path = 0.35 * lab14_np.sin(
    2 * lab14_np.pi * lab14_np.arange(lab14_action_n) / lab14_action_n)
lab14_action_reference_path -= lab14_action_reference_path.mean()


def lab14_action_parts(path):
    """Kinetic and potential contributions for the periodic oscillator path."""
    differences = lab14_np.roll(path, -1) - path
    kinetic = lab14_action_m * lab14_np.sum(differences**2) / (2 * lab14_action_dt)
    potential = (lab14_action_dt * lab14_action_m * lab14_action_omega**2
                 * lab14_np.sum(path**2) / 2)
    return kinetic, potential


# Both phases return to, and are compared with, this same fixed reference.
lab14_action_reference_parts = lab14_np.array(
    lab14_action_parts(lab14_action_reference_path))
lab14_action_reference_total = lab14_action_reference_parts.sum()
lab14_action_profile = lab14_np.sin(lab14_np.linspace(0, lab14_np.pi, 25))
lab14_action_profile[[0, -1]] = 0.0
lab14_action_offsets = lab14_np.r_[1.1 * lab14_action_profile,
                                0.35 * lab14_action_profile]
lab14_action_phase_names = (['Move one point'] * 25
                           + ['Shift the whole path'] * 25)
lab14_action_paths = lab14_np.tile(lab14_action_reference_path, (50, 1))
lab14_action_paths[:25, lab14_action_index] += lab14_action_offsets[:25]
lab14_action_paths[25:] += lab14_action_offsets[25:, None]
lab14_action_components = lab14_np.array(
    [lab14_action_parts(path) for path in lab14_action_paths])
lab14_action_kinetic = lab14_action_components[:, 0]
lab14_action_potential = lab14_action_components[:, 1]
lab14_action_total = lab14_action_components.sum(axis=1)
lab14_action_relative_weight = lab14_np.exp(
    -(lab14_action_total - lab14_action_reference_total) / lab14_action_hbar)

lab14_action_figure = lab14_plt.figure(figsize=(10, 2.8), dpi=100, layout='none')
lab14_action_grid = lab14_action_figure.add_gridspec(
    2, 2, width_ratios=(1.85, 1), height_ratios=(3.2, 1),
    left=0.075, right=0.98, bottom=0.18, top=0.88, wspace=0.34, hspace=0.65)
lab14_action_path_ax = lab14_action_figure.add_subplot(lab14_action_grid[:, 0])
lab14_action_bar_ax = lab14_action_figure.add_subplot(lab14_action_grid[0, 1])
lab14_action_weight_ax = lab14_action_figure.add_subplot(lab14_action_grid[1, 1])
lab14_action_weight_ax.set_axis_off()
lab14_action_closed_reference = lab14_np.r_[lab14_action_reference_path,
                                           lab14_action_reference_path[0]]
lab14_action_path_ax.plot(
    lab14_action_tau, lab14_action_closed_reference, '--',
    color='#707070', lw=2, label='Fixed reference', zorder=1)
lab14_action_line, = lab14_action_path_ax.plot(
    lab14_action_tau, lab14_action_closed_reference, 'o-',
    color='#007c91', ms=3, lw=2, label='Current path', zorder=2)
lab14_action_link_line, = lab14_action_path_ax.plot(
    [], [], 'o-', color='#b84b20', ms=5, lw=3, zorder=3)
lab14_action_path_ax.set(
    xlim=(0, lab14_action_tau[-1]),
    ylim=(lab14_action_paths.min() - 0.18, lab14_action_paths.max() + 0.18),
    xlabel='Imaginary time τ', ylabel='Position x')
lab14_action_title = lab14_action_path_ax.set_title('Move one point', fontsize=14)
lab14_action_path_ax.legend(loc='upper left', fontsize=12, frameon=False)
lab14_action_bars = lab14_action_bar_ax.bar(
    [0, 1], lab14_action_reference_parts / lab14_action_hbar,
    color=['#007c91', '#b84b20'], width=0.55)
for lab14_action_bar_index, lab14_action_ref_value in enumerate(
        lab14_action_reference_parts / lab14_action_hbar):
    lab14_action_bar_ax.plot(
        [lab14_action_bar_index - 0.36, lab14_action_bar_index + 0.36],
        [lab14_action_ref_value] * 2, '--', color='#707070', lw=1.5)
lab14_action_bar_ax.set(
    xticks=[0, 1], xticklabels=['Kinetic / ħ', 'Potential / ħ'],
    ylim=(0, 1.16 * lab14_action_components.max() / lab14_action_hbar),
    ylabel='Action')
lab14_action_bar_ax.set_title('Action contributions', fontsize=12)
lab14_action_bar_ax.set_yticks([0, 1, 2, 3])
lab14_action_weight_text = lab14_action_weight_ax.text(
    0.5, 0.75, '', transform=lab14_action_weight_ax.transAxes,
    ha='center', va='center', fontsize=13)
lab14_action_weight_ax.text(
    0.5, 0.05, r'$w/w_{\rm ref}=e^{-(S-S_{\rm ref})/\hbar}$',
    transform=lab14_action_weight_ax.transAxes,
    ha='center', va='center', fontsize=12)
for lab14_action_axis in (lab14_action_path_ax, lab14_action_bar_ax):
    lab14_action_axis.tick_params(labelsize=12)
    lab14_action_axis.xaxis.label.set_size(12)
    lab14_action_axis.yaxis.label.set_size(12)


def lab14_action_update(frame):
    """Draw one precomputed deformation without touching sampler state."""
    path = lab14_action_paths[frame]
    lab14_action_line.set_ydata(lab14_np.r_[path, path[0]])
    lab14_action_title.set_text(lab14_action_phase_names[frame])
    if frame < 25:
        nearby = lab14_np.arange(lab14_action_index - 1, lab14_action_index + 2)
        lab14_action_link_line.set_data(nearby * lab14_action_dt, path[nearby])
    else:
        lab14_action_link_line.set_data([], [])
    for bar, contribution in zip(lab14_action_bars, lab14_action_components[frame]):
        bar.set_height(contribution / lab14_action_hbar)
    lab14_action_weight_text.set_text(
        f'Relative weight = {lab14_action_relative_weight[frame]:.3f}')
    return (lab14_action_line, lab14_action_link_line, lab14_action_title,
            lab14_action_weight_text, *lab14_action_bars)


lab14_action_update(0)
lab14_action_animation = lab14_FuncAnimation(
    lab14_action_figure, lab14_action_update, frames=50,
    interval=110, repeat=False, blit=False)
# JSHTML starts paused. "Once" is the selected playback mode.
# Frame export can re-read layout defaults; preserve this explicit GridSpec.
with lab14_plt.rc_context({'figure.constrained_layout.use': False, 'figure.autolayout': False}):
    lab14_action_animation_html = lab14_action_animation.to_jshtml(default_mode='once')
# Matplotlib adds a remote icon stylesheet; native symbols keep this output offline.
lab14_action_animation_html = lab14_re.sub(
    r'<link\b[^>]*>', '', lab14_action_animation_html)
lab14_action_control_symbols = {
    'fa fa-fast-backward': '⏮', 'fa fa-step-backward': '‹',
    'fa fa-play fa-flip-horizontal': '◀', 'fa fa-pause': 'Ⅱ',
    'fa fa-play': '▶', 'fa fa-step-forward': '›',
    'fa fa-fast-forward': '⏭', 'fa fa-minus': '−', 'fa fa-plus': '+',
}
for lab14_action_icon, lab14_action_symbol in lab14_action_control_symbols.items():
    lab14_action_animation_html = lab14_action_animation_html.replace(
        f'<i class="{lab14_action_icon}"></i>',
        f'<span style="font-size:16px">{lab14_action_symbol}</span>')
lab14_plt.close(lab14_action_figure)
lab14_display(lab14_HTML(
    "<div class='lab14-path-animation'>"
    "<style>.lab14-path-animation .anim-buttons{margin:3px 0}"
    ".lab14-path-animation .anim-state{font-size:12px}"
    ".lab14-path-animation .anim-buttons button{height:22px}</style>"
    + lab14_action_animation_html
    + "<p style='font-size:14px;margin:3px 0'>Dashed marks show the fixed reference; "
      "orange highlights the two affected links.</p></div>"))

We're choosing these changes to show how the formula works. The displayed weight compares each curve with the starting curve: $e^{-(S_E-S_{\rm ref})/\hbar}$. Shifting the whole curve preserves the differences between neighbors.

## Remember flipping one spin? Here we move one point.

Propose $x_i'=x_i+u$, with $u$ uniform on $[-h,h]$.

One Ising spin flip changed four neighboring bonds. Moving one path coordinate changes the two links touching it. That's all we need to recompute.

In [ ]:
i = n_tau // 2
proposal = example_path.copy()
proposal[i] += 0.65
fig, ax = plt.subplots()
ax.plot(tau_grid, np.r_[example_path, example_path[0]], 'o-',
        color='#172b4d', label='Current path')
ax.plot(tau_grid, np.r_[proposal, proposal[0]], 'o--',
        color='#a54632', label='Proposed path')
ax.plot(tau_grid[i-1:i+2], proposal[i-1:i+2], color='#a54632', lw=3)
ax.annotate('One coordinate changes', (tau_grid[i], proposal[i]),
            xytext=(i*dt+0.3, proposal[i]+0.18),
            arrowprops={'arrowstyle': '->'}, fontsize=10)
ax.set(xlim=((i - 3) * dt, (i + 3) * dt),
       ylim=(-0.65, 1.05), xlabel=r'Imaginary time $\tau$', ylabel='Position x')
ax.legend(loc='lower left', fontsize=10)
plt.show()

Subtract the old contributions from the proposed contributions to get $\Delta S_E$. The rest of the path cancels.

## Compute the local action change

Use the same shortcut we used for a spin flip: add the changed terms and subtract their old values. Here that means two links.

In [ ]:
@njit
def delta_action(path, x_prime, i, dt):
    left = path[(i - 1) % path.size]
    right = path[(i + 1) % path.size]
    new = action(left, x_prime, dt) + action(x_prime, right, dt)
    old = action(left, path[i], dt) + action(path[i], right, dt)
    return new - old

This works at $i=0$ and at the last stored point because both neighbor indices wrap around.

## Check the shortcut

Calculate the change both ways: from the two affected links and from the full path. The results should agree, including across the closing link.

In [ ]:
local_change = delta_action(example_path, proposal[i], i, dt)
full_change = total_action(proposal, dt) - total_action(example_path, dt)
print(f"Local change: {local_change:.8f}")
print(f"Full change:  {full_change:.8f}")
np.testing.assert_allclose(local_change, full_change, atol=1e-12)

This gives us a direct check of the local action calculation before we use it thousands of times.

## Use the same Metropolis rule

Remember accepting spin flips using $e^{-\Delta E/(k_BT)}$? Here the exponent is $-\Delta S_E/\hbar$:

$$A=\min\left(1,e^{-\Delta S_E/\hbar}\right).$$

Accept every move that lowers the action. Sometimes accept a move that raises it. After a rejection, keep the current path and count the attempt.

## Put one update into code

Choose a proposed position, calculate the action change, and accept or reject. The proposed position can lie anywhere on the real line.

In [ ]:
@njit
def update_point(path, i, dt, hitsize, rng):
    x_prime = path[i] + rng.uniform(-hitsize, hitsize)
    change = delta_action(path, x_prime, i, dt)
    if change <= 0 or rng.random() < np.exp(-change / HBAR):
        path[i] = x_prime
        return 1
    return 0

The plotting window just sets what we can see. The update uses the full proposed position when calculating the acceptance probability.

## Make a sweep through the path

For Ising, a sweep meant as many random spin-flip attempts as there were spins. Here we visit every path coordinate once, in order. Each update uses the path as it stands after the previous update.

In [ ]:
@njit
def sweep(path, dt, hitsize, rng):
    accepted = 0
    for i in range(path.size):
        accepted += update_point(path, i, dt, hitsize, rng)
    return accepted / path.size

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_sweep_rng = np.random.default_rng(1403)
lab14_before_sweep = example_path.copy()
lab14_after_one = lab14_before_sweep.copy()
lab14_first_accepted = update_point(lab14_after_one, 0, dt, 1.0, lab14_sweep_rng)
lab14_after_sweep = lab14_after_one.copy()
lab14_sweep_accepts = lab14_first_accepted
for point in range(1, n_tau):
    lab14_sweep_accepts += update_point(lab14_after_sweep, point, dt, 1.0, lab14_sweep_rng)
fig, axes = plt.subplots(1, 3, figsize=(11.2, 2.25), sharex=True, sharey=True)
stages = [(lab14_before_sweep, 'Starting path'),
          (lab14_after_one, 'After 1 attempt: ' + ('accepted' if lab14_first_accepted else 'rejected')),
          (lab14_after_sweep, f'After 1 sweep: {lab14_sweep_accepts}/{n_tau} accepted')]
for ax, (path, title) in zip(axes, stages):
    ax.plot(tau_grid, np.r_[lab14_before_sweep, lab14_before_sweep[0]],
            color='#b4c2d0', lw=1.5)
    ax.plot(tau_grid, np.r_[path, path[0]], 'o-', color='#007c91', ms=2.5)
    ax.set(title=title, xlabel=r'Imaginary time $\tau$')
axes[0].set_ylabel('Position x')
plt.show()

Gray shows the starting path. One attempt can change one coordinate; a full sweep attempts every coordinate. Save the whole resulting path as one sample.

## Let the chain warm up, then save paths

Remember starting the Ising lattice with all spins up? We discarded the initial relaxation before measuring. Here we start every position at zero and discard the warm-up sweeps.

In [ ]:
@njit
def sample_paths(n_tau, dt, hitsize, burn_in, sweeps, rng):
    path = np.zeros(n_tau)
    warmup = np.empty(burn_in)
    samples = np.empty((sweeps, n_tau))
    acceptance = np.empty(sweeps)
    for k in range(burn_in):
        sweep(path, dt, hitsize, rng)
        warmup[k] = np.mean(path**2)
    for k in range(sweeps):
        acceptance[k] = sweep(path, dt, hitsize, rng)
        samples[k] = path
    return warmup, samples, acceptance

Each row of `samples` holds one saved path. Copying the coordinates into that row preserves the sample while the chain continues.

## Start with a coarse time grid

Use $\Delta\tau=1$ and total imaginary time 30. This gives us 30 stored coordinates per path. We'll check the result, then repeat with a smaller time step.

In [ ]:
DT = 1.0
NTAU = int(round(TAU / DT))
HITSIZE = 1.0
BURN_IN = 10_000
SWEEPS = 40_000
warmup, samples, acceptance = sample_paths(
    NTAU, DT, HITSIZE, BURN_IN, SWEEPS, np.random.default_rng(142))
x2_trace = np.mean(samples**2, axis=1)
print(f"{samples.shape[0]:,} paths; {samples.shape[1]} points per path")
print(f"Accepted fraction: {acceptance.mean():.3f}")

The first call includes compilation. Each run starts from the stated seed and collects a fresh set of measurements.

## Watch a measurement change from sweep to sweep

Remember plotting the mean magnetization after each Ising sweep? Here we measure the mean-square position of each path:

$$\overline{x^2}=\frac{1}{N_\tau}\sum_i x_i^2.$$

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(BURN_IN), warmup, lw=0.5, color="#a54632", label="Discarded")
ax.plot(BURN_IN + np.arange(SWEEPS), x2_trace,
        lw=0.5, color="#007c91", label="Retained")
ax.axvline(BURN_IN, color="black", ls="--", lw=1)
ax.set(xlabel="Sweep", ylabel=r"Path mean of $x^2$")
ax.legend()
plt.show()

The horizontal axis counts Monte Carlo sweeps. Look for the initial relaxation and for stretches where successive measurements move together. Those correlations matter when we estimate the uncertainty.

## Build the histogram from one, a few, and many samples

These are now the oscillator paths from our MCMC run. The top row shows the saved paths. The bottom row collects their positions into a histogram.

In [ ]:
display(HTML('<span class="lab14-visual-marker" hidden></span>'))
lab14_counts = [1, 5, len(samples)]
lab14_limit = np.ceil(np.abs(samples).max()*2)/2
lab14_bins = np.linspace(-lab14_limit, lab14_limit, 41)
fig, axes = plt.subplots(2, 3, figsize=(11.2, 3.8), sharex='row', sharey='row')
for col, count in enumerate(lab14_counts):
    retained = samples[:count]
    drawn = np.linspace(0, count-1, min(count, 100), dtype=int)
    for index in drawn:
        axes[0, col].plot(np.arange(NTAU+1)*DT,
            np.r_[retained[index], retained[index, 0]],
            color='#007c91', lw=1, alpha=0.15 if count > 100 else 0.7)
    title = f'{count:,} saved path' + ('s' if count > 1 else '')
    if count > 100:
        title += '\n100 drawn; all used below'
    axes[0, col].set(title=title, xlabel=r'Imaginary time $\tau$',
                     ylim=(-lab14_limit, lab14_limit))
    axes[1, col].hist(retained.ravel(), bins=lab14_bins, density=True,
                     color='#9fc9d3', edgecolor='white', linewidth=0.4)
    axes[1, col].set(xlabel='Position x', xlim=(-lab14_limit, lab14_limit))
    axes[1, col].text(0.97, 0.91, f'{retained.size:,} coordinates',
                     transform=axes[1, col].transAxes, ha='right', va='top', fontsize=10)
axes[0, 0].set_ylabel('Position x')
axes[1, 0].set_ylabel('Position density')
plt.show()

Each saved sample contains a whole path. Adding more paths fills out the position histogram we wanted at the start. Positions within a path and successive paths are correlated; we'll use blocking to estimate the uncertainty.

## What answer should this time grid give?

Approximating a continuous path with a finite time step changes the prediction. For a long oscillator path with this action,

$$\langle x^2\rangle_{\Delta\tau}\simeq
\frac{\hbar}{m\omega\sqrt{4+(\omega\Delta\tau)^2}}.$$

At $\Delta\tau=1$, we expect **0.4472**. The continuum ground-state value is **0.5**.

In [ ]:
def lattice_variance(n_tau, dt):
    modes = np.arange(n_tau)
    eigenvalues = dt * OMEGA**2 + 4 / dt * np.sin(np.pi * modes / n_tau)**2
    return HBAR / M * np.mean(1 / eigenvalues)

variance_dt = lattice_variance(NTAU, DT)
print(f"Finite-grid prediction: {variance_dt:.6f}")

This normal-mode sum gives the exact prediction for our finite path. It tells us what the sampler should produce at this time step.

## Compare the position densities

Compare the histogram with the finite-grid prediction to check the sampling. Compare the finite-grid prediction with the continuum curve to see the effect of the time step.

In [ ]:
rho_lattice = np.exp(-x_grid**2 / (2 * variance_dt)) / np.sqrt(
    2 * np.pi * variance_dt)
fig, ax = plt.subplots()
ax.hist(samples.ravel(), bins=100, density=True, color="#9fc9d3", label="MCMC")
ax.plot(x_grid, rho_lattice, color="#172b4d", lw=2, label="Finite grid")
ax.plot(x_grid, rho_ground, "--", color="#a54632", lw=2, label="Continuum")
ax.set(xlim=(-3, 3), xlabel="Position x", ylabel="Probability density")
ax.legend()
plt.show()

At this time step, we expect a narrower histogram. More sweeps reduce the sampling noise. A smaller time step brings the grid prediction toward the continuum answer.

## Remember blocking the Ising measurements?

Successive magnetization measurements were correlated. We averaged consecutive measurements into blocks, then estimated the uncertainty from the block averages. Do the same with the mean-square positions.

In [ ]:
def blocked_mean_error(values, block_size):
    n_blocks = values.size // block_size
    blocks = values[:n_blocks * block_size].reshape(n_blocks, block_size)
    means = blocks.mean(axis=1)
    return values.mean(), means.std(ddof=1) / np.sqrt(n_blocks)

for block_size in (250, 500, 1000, 2000):
    mean, error = blocked_mean_error(x2_trace, block_size)
    print(f"Block {block_size:4d}: <x²> = {mean:.4f} ± {error:.4f}")

Increase the block size until the error estimate settles, while keeping enough blocks to estimate it. If it keeps growing, collect a longer chain.

## Now make the time grid finer

Keep the total imaginary time fixed, so the temperature stays fixed. Halving the time step doubles the number of coordinates in each path.

In [ ]:
time_steps = np.array([1.0, 0.5, 0.25])
means, errors = [blocked_mean_error(x2_trace, 1000)[0]], [
    blocked_mean_error(x2_trace, 1000)[1]]
for run, smaller_dt in enumerate(time_steps[1:], start=1):
    smaller_n = int(round(TAU / smaller_dt))
    _, fine_samples, _ = sample_paths(
        smaller_n, smaller_dt, HITSIZE * np.sqrt(smaller_dt),
        BURN_IN, SWEEPS, np.random.default_rng(142 + run))
    mean, error = blocked_mean_error(np.mean(fine_samples**2, axis=1), 1000)
    means.append(mean)
    errors.append(error)

Scale the proposal with $\sqrt{\Delta\tau}$ to keep the typical kinetic-action change comparable. As with Ising, check how quickly the measurements decorrelate.

## Which error are we reducing?

The error bars show sampling uncertainty. The curve shows how the finite time step shifts the expected answer. Look at what changes as we make the grid finer.

In [ ]:
dt_curve = np.linspace(0, 1.05, 200)
variance_curve = HBAR / (M * OMEGA * np.sqrt(4 + (OMEGA * dt_curve)**2))
fig, ax = plt.subplots()
ax.plot(dt_curve, variance_curve, color="#172b4d", label="Long-path grid prediction")
ax.axhline(0.5, color="#a54632", ls="--", label="Continuum ground state")
ax.errorbar(time_steps, means, yerr=errors, fmt="o", capsize=4,
            color="#007c91", label="MCMC ± blocked error")
ax.set(xlabel=r"Time step $\Delta\tau$", ylabel=r"$\langle x^2\rangle$")
ax.legend(fontsize=10)
plt.show()

Agreement with the grid prediction checks the sampler. Moving toward 0.5 checks convergence as we reduce the time step. The finer chain is more correlated, so its blocked error estimate is noisier.

## What should we change in the next run?

| What we want to improve | What to change |
|---|---|
| Sampling uncertainty | Collect more equilibrated sweeps |
| Dependence on the starting path | Discard more warm-up and compare starts |
| Slow exploration | Adjust the proposal size and check correlations |
| Time-step error | Reduce $\Delta\tau$ at fixed total time |
| Excited-state contributions | Increase the total imaginary time |

Remember checking both acceptance and the magnetization trace? Check both acceptance and the measurement trace here too.

## Use this in Assignment 3

Use the assignment's total imaginary time and action. At finite total time, the continuum thermal prediction is

$$\langle x^2\rangle=\frac{\hbar}{2m\omega}\coth\!\left(\frac{\omega\tau_{\rm total}}2\right).$$

Report the time step, number of sweeps, discarded warm-up, and uncertainty method. Compare with the finite-grid prediction, then check convergence toward the continuum result.

## Same sampling workflow, a new physical system

1. Define a configuration: the whole path.
2. Calculate its statistical weight from the action.
3. Propose a local change and accept or reject it.
4. Warm up, save configurations, and measure.
5. Account for correlations and compare with the known answer.

That's the workflow we used for Ising. Here it lets us calculate the oscillator's position density.

Adapted from Javier Duarte's Lab 14 and the Mittal et al. example discussed in lecture.